In [11]:
import pandas as pd
from ete3 import Tree

In [12]:
three_data = "/home/ria/HSE/lab/model/knownpass/h1pdm.l.r.newick"
aligned_data = "/home/ria/HSE/lab/model/knownpass/h1pdm.all.fa"
out_data = "/home/ria/HSE/lab/new_three/data/my_output/h1_test.csv"

1. Достаю названия узлов из ancestral_sequences и делаю словарь название - последовательность

In [13]:
def read_fasta(path):
    seqs = {}
    name = None
    buf = []

    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if name is not None:
                    seqs[name] = "".join(buf)
                name = line[1:]   
                buf = []
            else:
                buf.append(line)

        if name is not None:
            seqs[name] = "".join(buf)

    return seqs

In [14]:
seqs = read_fasta(aligned_data)

In [15]:
for i, name in enumerate(seqs):
    print(name)
    if i == 5:
        break


STRAIN255081_2009
STRAIN211474_2009
STRAIN277461_2009
STRAIN212929_2009
STRAIN246884_2009
STRAIN256502_2009


2. разбиваем последоваательность на сайты, то есть мы из фасты узнаем какие по счету нуклеотиды входят в какие сайты

In [16]:
def get_codon(seq, site):
    i = (site - 1) * 3
    return seq[i:i+3]


3. загрузка дерева
   

In [17]:
t = Tree(three_data, format=1)

In [18]:
# проверка, что для каждого узла есть последовательность
missing = [] #тут будут хранится узлы с проблемами
for n in t.traverse():
    if not n.name: # все ли узлы с именем
        missing.append("(unnamed node)")
    elif n.name not in seqs: #все ли узлы есть в словаре
        missing.append(n.name)


In [19]:
print(missing)

['(unnamed node)', '(unnamed node)']


я эту проверку раньше не делала, только сейчас подумала

разрешаем отсутствие последовательности у корневого узла, но если для любого другого узла дерева нет последовательности в словаре, анализ немедленно прекращается

In [20]:
missing = [m for m in missing if m != "root"]

if missing:
    raise ValueError(
        "Нет последовательностей для узлов (первые 20):\n"
        + "\n".join(missing[:20])
        + "\n\nДерево не согласовано с ancestral_sequences."
    )


ValueError: Нет последовательностей для узлов (первые 20):
(unnamed node)
(unnamed node)

Дерево не согласовано с ancestral_sequences.

Берем одну последоватеьность из списка и проверяем, что последовательность кодонная (длина кратна 3), и вычисляем число кодонных сайтов, по которым дальше будет вестись анализ.

In [21]:

any_seq = next(iter(seqs.values()))
if len(any_seq) % 3 != 0:
    raise ValueError("Длина последовательности не кратна 3")
n_sites = len(any_seq) // 3

n_sites

566

Идет от листа к корню и собирает замены кодонов по пути. Возвращает список событий.


In [ ]:
def walk_up_from_leaf(leaf):
  
    events = [] #список с найденными цепочками
    current = leaf #стартуем с выбранного листа
    current_seq = seqs.get(current.name) # берем последовательность этого листа из словаря seqs

    while current.up is not None: # углубляемся внутрь дерева, пока у узла есть родитель
        parent = current.up

        # на всякий случай если current_seq is None или parent.name not in seqs, то мы не можем сравнивать кодоны и продвигаемся дальше

        if current_seq is None or parent.name not in seqs:
            current = parent
            current_seq = seqs.get(current.name)
            continue

        parent_seq = seqs[parent.name] #получили последовательность родителя
        grandparent = parent.up #и дедушки
        grandparent_seq = seqs.get(grandparent.name) if (grandparent is not None) else None #может не быть если родитель - корень
        
        # получаем дистанцию
        dist = t.get_distance(parent.name, current.name)

        # перебираем все сайты и достаем кодоны у предка и потомка
        for site in range(1, n_sites + 1):
            der_codon = get_codon(current_seq, site)  
            anc_codon = get_codon(parent_seq, site)   

            if der_codon == anc_codon: 
                continue
            
            # достаем кодон дедушки
            if grandparent_seq is not None:
                grandanc_codon = get_codon(grandparent_seq, site)
            else:
                grandanc = None

            events.append({
                    "site": site,
                    "ancestor_node": parent.name,     # узел замены родителя (anc_codon)
                    "grandanc_codon": grandanc_codon,  
                    "anc_codon": anc_codon,            
                    "der_codon": der_codon,                  
                    "distance": dist
             })

        # поднимаемся выше
        current = parent
        current_seq = parent_seq

    return events


In [ ]:
rows = []
for leaf in t.iter_leaves():
    rows.extend(walk_up_from_root(leaf))

df = pd.DataFrame(rows)

от корня к листам

In [31]:
def walk_up_from_root_fixed(tree, seqs, seq_length):
    """
    Обход дерева preorder. Для каждого сайта запоминаем последнюю замену
    на пути от корня. Когда встречаем новую замену в том же сайте —
    записываем пару (предыдущая замена → текущая замена).
    """
    results = []

    def get_codon(seq, site):
        """Извлекает кодон для сайта (1-based)."""
        idx = (site - 1) * 3
        return seq[idx:idx+3]

    # Рекурсивный обход preorder с "памятью" по каждому сайту
    def traverse(node, last_event):
        """
        last_event: dict {site: (grandanc_codon, grandanc_aa, anc_codon, anc_aa, anc_node_name)}
        — для каждого сайта хранит информацию о последней замене на пути от корня к текущему узлу.
        grandanc = состояние ДО той замены, anc = состояние ПОСЛЕ той замены.
        """
        parent = node.up
        if parent is None:
            # корень — просто идём в детей
            for child in node.children:
                traverse(child, last_event.copy())
            return

        parent_seq = seqs.get(parent.name)
        node_seq = seqs.get(node.name)

        if parent_seq is None or node_seq is None:
            for child in node.children:
                traverse(child, last_event.copy())
            return

        n_sites = seq_length // 3

        for site in range(1, n_sites + 1):
            anc_codon = get_codon(parent_seq, site)
            der_codon = get_codon(node_seq, site)

            # Транслируем
            anc_aa = translate_codon(anc_codon)
            der_aa = translate_codon(der_codon)

            # Пропускаем если нет замены на аминокислотном уровне
            if anc_aa is None or der_aa is None or anc_aa == der_aa:
                continue

            # На ребре parent→node в этом сайте есть замена anc_aa → der_aa
            # Проверяем, была ли предыдущая замена на этом пути
            if site in last_event:
                prev = last_event[site]
                # Расстояние = длина ветви от предыдущей замены до текущей
                # (нужно будет подставить свой расчёт distance)
                results.append({
                    'site': site,
                    'ancestor_node': parent.name,       # родитель текущего ребра
                    'grandanc_codon': prev['grandanc_codon'], # состояние ДО предыдущей замены
                    'grandanc_aa': prev['grandanc_aa'],
                    'anc_codon': prev['anc_codon'],           # состояние ПОСЛЕ предыдущей замены
                    'anc_aa': prev['anc_aa'],
                    'der_codon': der_codon,                   # состояние ПОСЛЕ текущей замены
                    'der_aa': der_aa,
                    'distance': node.dist  # <-- тут нужен правильный расчёт distance
                })

            # Обновляем last_event для этого сайта:
            # теперь "последняя замена" — это текущая
            last_event[site] = {
                'grandanc_codon': anc_codon,  # состояние ДО текущей замены
                'grandanc_aa': anc_aa,
                'anc_codon': der_codon,        # состояние ПОСЛЕ текущей замены
                'anc_aa': der_aa,
                'anc_node': node.name
            }

        for child in node.children:
            traverse(child, last_event.copy())

    traverse(tree, {})
    return results

In [32]:
events = walk_up_from_root(t, seqs, n_sites)
print(f" Собрано событий замен: {len(events)}")
df = pd.DataFrame(events)

 Собрано событий замен: 12968


In [24]:
# сохраняем
df.to_csv("/home/ria/HSE/lab/new_three/data/my_output/h1pdm_test_rename.сsv", sep=",", index=False)

In [33]:
CODON2AA = {
    # Phenylalanine 
    "TTT":"F","TTC":"F",
    # Leucine
    "TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    # Isoleucine 
    "ATT":"I","ATC":"I","ATA":"I",
    # Methionine
    "ATG":"M",
    # Valine
    "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    # Serine
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
    # Proline 
    "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
    # Threonine 
    "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
    # Alanine 
    "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
    # Tyrosine 
    "TAT":"Y","TAC":"Y",
    # Histidine 
    "CAT":"H","CAC":"H",
    # Glutamine
    "CAA":"Q","CAG":"Q",
    # Asparagine 
    "AAT":"N","AAC":"N",
    # Lysine
    "AAA":"K","AAG":"K",
    # Aspartate 
    "GAT":"D","GAC":"D",
    # Glutamate
    "GAA":"E","GAG":"E",
    # Cysteine 
    "TGT":"C","TGC":"C",
    # Tryptophan
    "TGG":"W",
    # Arginine
    "CGT":"R","CGC":"R","CGA":"R","CGG":"R",
    "AGA":"R","AGG":"R",
    # Serine 
    "AGT":"S","AGC":"S",
    # Glycine
    "GGT":"G","GGC":"G","GGA":"G","GGG":"G",
    # Stop
    "TAA":"Stop","TAG":"Stop","TGA":"Stop",
}


переводим

In [26]:
def codon_to_aa(c):
    if c is None:
        return None
    c = str(c).upper().replace("U", "T")
    if len(c) != 3:
        return None
    # если есть не-типичный нк
    if any(x not in "ATCG" for x in c):
        return "X"   # неизвестная аминокислота
    return CODON2AA.get(c, "X")


In [27]:
df["grandanc_aa"] = df["grandanc_codon"].map(codon_to_aa)
df["anc_aa"]      = df["anc_codon"].map(codon_to_aa)
df["der_aa"]      = df["der_codon"].map(codon_to_aa)


In [28]:
df.to_csv("/home/ria/HSE/lab/new_three/data/my_output/h1pdm_test_rename_aa.сsv", sep=",", index=False)


In [34]:
# оставляем только строки, где аминокислота  меняется
df_nonsyn = df[df["anc_aa"] != df["der_aa"]].copy()

print("До фильтрации:", len(df))
print("После:", len(df_nonsyn))



KeyError: 'anc_aa'

In [30]:
# оставляем только строки, где аминокислота  меняется
df_nonsyn_grand = df_nonsyn[df_nonsyn["grandanc_aa"] != df_nonsyn["anc_aa"]].copy()

print("До фильтрации:", len(df_nonsyn))
print("После:", len(df_nonsyn_grand))

До фильтрации: 4446
После: 6


In [105]:
df_nonsyn.to_csv(
    "/home/ria/HSE/lab/new_three/data/my_output/h1pdm_test_rename_aa_filtered.csv",
    sep=",",
    index=False
)


In [106]:
dup_mask = df_nonsyn.duplicated(keep="first")
print("Всего строк:", len(dup_mask))
print("Полных дублей (повторов строк):", dup_mask.sum())


Всего строк: 4446
Полных дублей (повторов строк): 0


In [107]:
df_nonsyn["grandanc_codon"].isna().sum()
df_nonsyn[df_nonsyn["grandanc_codon"].isna()]



,site,ancestor_node,grandanc_codon,anc_codon,der_codon,distance,grandanc_aa,anc_aa,der_aa


In [108]:
df[["grandanc_codon", "anc_codon", "der_codon"]].isna().sum()


grandanc_codon    0
anc_codon         0
der_codon         0
dtype: int64